# 01 — Yield Curve Data Retrieval

Download daily yield curve data for three regions:
- **UK**: Bank of England nominal spot curve (GLC dataset)
- **US**: FRED Treasury constant-maturity rates
- **EU**: ECB AAA-rated government bond spot rates

Maturities: 1Y, 2Y, 3Y, 5Y, 10Y, 20Y per region.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_retrieval import (
    fetch_uk_yields,
    fetch_us_yields,
    fetch_eu_yields,
    align_yield_data,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

START = '2000-01-01'
END = '2026-04-30'

## 1. UK Gilts (Bank of England)

In [ ]:
uk = fetch_uk_yields(start_date=START, end_date=END, force_download=False)
print(f"UK yields: {uk.shape}")
print(f"Date range: {uk.index.min()} to {uk.index.max()}")
print(f"NaN counts:\n{uk.isna().sum()}")
uk.head()

## 2. US Treasuries (FRED)

Requires `FRED_API_KEY` environment variable. Get a free key at:
https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
us = fetch_us_yields(start_date=START, end_date=END, force_download=False)
print(f"US yields: {us.shape}")
print(f"Date range: {us.index.min()} to {us.index.max()}")
print(f"NaN counts:\n{us.isna().sum()}")
us.head()

## 3. EU AAA-Rated Yields (ECB)

In [ ]:
eu = fetch_eu_yields(start_date=START, end_date=END, force_download=False)
print(f"EU yields: {eu.shape}")
print(f"Date range: {eu.index.min()} to {eu.index.max()}")
print(f"NaN counts:\n{eu.isna().sum()}")
eu.head()

## 4. Align and Merge

In [ ]:
combined = align_yield_data(uk, us, eu)
print(f"Combined shape: {combined.shape}")
print(f"Date range: {combined.index.min()} to {combined.index.max()}")
print(f"Remaining NaN: {combined.isna().sum().sum()}")
combined.describe()

## 5. Visualization — 10Y Yields Across Regions

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# 10Y yields
ax = axes[0]
for col, label in [('UK_10Y', 'UK'), ('US_10Y', 'US'), ('EU_10Y', 'EU')]:
    if col in combined.columns:
        ax.plot(combined.index, combined[col], label=label, alpha=0.8)
ax.set_ylabel('Yield (%)')
ax.set_title('10-Year Government Bond Yields')
ax.legend()

# 1Y yields
ax = axes[1]
for col, label in [('UK_1Y', 'UK'), ('US_1Y', 'US'), ('EU_1Y', 'EU')]:
    if col in combined.columns:
        ax.plot(combined.index, combined[col], label=label, alpha=0.8)
ax.set_ylabel('Yield (%)')
ax.set_title('1-Year Government Bond Yields')
ax.legend()

plt.tight_layout()
plt.show()

## 6. Save Aligned Data

In [ ]:
combined.to_csv('../data/raw/aligned_yields.csv')
print('Saved aligned yields to data/raw/aligned_yields.csv')